# Primes.jl Demo

In [1]:
using Primes

Prime numbers are what make cryptography actually work. RSA depends entirely on the fact that multiplying two large primes together is easy, but factoring the result back into those primes is effectively impossible.

[Primes.jl](https://juliamath.github.io/Primes.jl/stable/api/) is Julia's dedicated prime number library. This notebook is a tour of its most useful functions.

## `isprime`

The most basic question: is this number prime?

In [2]:
isprime(7), isprime(10), isprime(104729)

(true, false, true)

Simple enough for small numbers. But for cryptographic use, primes need to be hundreds of digits long.

Testing primality naively (checking every divisor up to $\sqrt{n}$ ) is practically impossible that this scale.

`isprime` in Julia uses a combination of trial division for small factors and the Miller-Rabin primality test for large numbers, the latter I'll cover in depth in a later notebook.

## `primes` and `prime`

Though they both look and sound similar, these two functions answer two different questions:

- `primes(n)` gives all primes **up to** $n$.
- `prime(n)` gives the $n\text{th}$ prime.

In [3]:
primes(30)

10-element Vector{Int64}:
  2
  3
  5
  7
 11
 13
 17
 19
 23
 29

This is equivalent to running the **Sieve of Eratosthenes** up to $30$, a topic which deserves it's own dedicated notebook.

In [4]:
prime(30)

113

For anyone who's manually written the Sieve of Eratosthenes with an upper bound estimate knows how much cleaner that is.

`prime(n)` handles all of that internally.

## `factor`

`factor(n)` returns the complete prime factorization of $n$ as a `Factorization` object (essentially a dictionary mapping each prime factor to its exponent).

In [5]:
factor(12), factor(561), factor(600851475143)

(Primes.Factorization(2 => 2, 3 => 1), Primes.Factorization(3 => 1, 11 => 1, 17 => 1), Primes.Factorization(71 => 1, 839 => 1, 1471 => 1, 6857 => 1))

As touched on earlier, the reason factoring matters in cryptography is that it's a fundamentally difficult problem.

Given $n = pq$ in RSA where $p$ and $q$ are both large prime numbers, there is no known efficient algorithm for recovering $p$ and $q$ from $n$ alone. This is essentially what makes RSA secure.

## `nextprime` and `prevprime`

These return the nearest prime above or below a given value:

In [6]:
nextprime(100), prevprime(100)

(101, 97)

Note that these take a **value**, not an index.

`nextprime(100)` means "the first prime greater than $100$, not "the $100\text{th}$ prime". Compare:

In [7]:
prime(100), nextprime(100)

(541, 101)

In practice, `nextprime` is how you'd generate primes for RSA key generation.

Essentially: pick a random large number and find the nearest prime:

In [8]:
using Random 
seed = rand(big(2)^512 : big(2)^513) # random 512-bit number
p = nextprime(seed)

19040644755667225434866830467412419116929373974988494506431416296917840281856028016288713565286835448470754917108993278293467821001637223087014014996875849

That's a cryptographic-scale prime generated in one line.

In RSA, two primes not unlike the one above become $p$ and $q$.

## Some Bonus Useful Functions

A few more functions worth knowing:

### `totient`

This computes Euler's totient function $\phi(n)$ directly (see notebook on Fermat's and Euler's theorems for explanation of both).

*"How many integers in $1:n$ are coprime to $n$?"*

In [10]:
p, q = 61, 53
totient(p * q), (p-1)*(q-1) # should match

(3120, 3120)

### `eachfactor`

This iterates over prime, exponent pairs.

In [11]:
for (prime, exp) in eachfactor(360)
    println("$prime^$exp")
end

2^3
3^2
5^1


### `radical`

This returns the product of distinct prime factors.

In [17]:
radical(12) # 2^2 * 3 -> radical is 2 * 3 = 6

6

In [18]:
radical(360) # 2^3 * 3^2 * 5 -> radical is 2 * 3 * 5 = 30

30

### `ismersenneprime`

Checks if $n$ is of the form $2^p - 1$.

In [20]:
ismersenneprime(7) # 2^3 - 1 = 7 = true

true

In [23]:
ismersenneprime(31) # 2^5 - 1 = 31 = true

true

In [25]:
ismersenneprime(15) # not prime (produces an error))

ArgumentError: ArgumentError: The argument given is not a valid Mersenne Number (`M = 2^p - 1`).

> Note: `ismersenneprime` throws an error if the input is not a valid Mersenne number of the form $2^p - 1$. It doesn't just return `false`.